# 09. Statistical Rigor: Winners vs Underperformers Cohorts
## Hypothesis Testing with Mann-Whitney U & Cliff's Delta
Comparing top 25% viral videos against bottom 25% underperforming videos:
- Non-parametric Mann-Whitney U test (handles skewness)
- Effect size estimation (Cliff's Delta / Rank Biserial)


In [ ]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
conn = sqlite3.connect(os.path.join(PROJECT_ROOT, "youtube_growth.db"))

df = pd.read_sql_query("""
    SELECT v.video_id, v.duration_seconds, s.views, s.average_view_percentage,
           m.virality_score, m.engagement_rate
    FROM videos v
    JOIN video_statistics s ON v.video_id = s.video_id
    JOIN derived_video_metrics m ON v.video_id = m.video_id
""", conn)

q75 = df['virality_score'].quantile(0.75)
q25 = df['virality_score'].quantile(0.25)

winners = df[df['virality_score'] >= q75]
underperformers = df[df['virality_score'] <= q25]

# Mann-Whitney U test on Retention Rate
u_stat, p_val = stats.mannwhitneyu(winners['average_view_percentage'], underperformers['average_view_percentage'], alternative='two-sided')

# Cliff's Delta calculation
n1, n2 = len(winners), len(underperformers)
greater = sum(w > u for w in winners['average_view_percentage'] for u in underperformers['average_view_percentage'])
lesser = sum(w < u for w in winners['average_view_percentage'] for u in underperformers['average_view_percentage'])
cliffs_delta = (greater - lesser) / (n1 * n2)

print(f"Winners Cohort (n={n1}) Mean Retention: {winners['average_view_percentage'].mean():.2f}%")
print(f"Underperformers Cohort (n={n2}) Mean Retention: {underperformers['average_view_percentage'].mean():.2f}%")
print(f"Mann-Whitney U Statistic: {u_stat:.1f}, p-value: {p_val:.5e}")
print(f"Cliff's Delta Effect Size: {cliffs_delta:.3f} (Values > 0.43 indicate large effect size)")
